<a href="https://colab.research.google.com/github/RubiPacheco/cheme-assistant/blob/main/cheme_assistant_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ChemE Assistant: an AI Agent with Tool Calling

An AI agent that answers chemical engineering questions. Instead of letting the language model guess numbers, the agent **decides which calculation tool to call**, runs real Python code, and explains the result in plain language.

**How it works**

1. The user asks a question in natural language (Spanish or English).
2. Gemini (the LLM) reads the question and the description of each available tool.
3. The model chooses a tool and its arguments (a *tool call*).
4. Python runs the function and returns the exact result.
5. The model explains the answer step by step.

**Tools**

| Tool | What it does |
|---|---|
| `convert_units` | Temperature, pressure and mass conversions |
| `ideal_gas_law` | Solves PV = nRT for P, V, n or T |
| `mixer_mass_balance` | Steady-state mass balance for a mixer |

**Stack:** Python · Google Gemini API · Google Colab

## 1. Setup

Get a free API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey). In Colab, open the 🔑 **Secrets** panel (left sidebar), add a secret named `GEMINI_API_KEY`, paste the key, and enable notebook access. The key is never written in the code.

In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [2]:
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# Google retires models over time, so instead of hard-coding one,
# pick the first "flash" model that actually answers for this account.
CANDIDATES = ["gemini-3.5-flash", "gemini-3-flash-preview", "gemini-flash-latest",
              "gemini-3.5-flash-lite"]

def pick_model():
    # Listing models is free; only test the ones that are listed, to save quota
    available = [m.name.replace("models/", "") for m in client.models.list()
                 if "flash" in m.name and "generateContent" in (m.supported_actions or [])]
    ordered = [c for c in CANDIDATES if c in available] + [a for a in available if a not in CANDIDATES]
    for name in ordered:
        try:
            client.models.generate_content(model=name, contents="Reply OK")
            return name
        except Exception as e:
            print(f"   {name}: not available ({str(e)[:60]}...)")
    raise RuntimeError("No flash model available. Check your API key in Secrets.")

MODEL = pick_model()
print(f"Client ready ✅  Using model: {MODEL}")

Client ready ✅  Using model: gemini-3.5-flash


## 2. Tools

Plain Python functions. The type hints and docstrings matter: the SDK turns them into the tool descriptions the model reads to decide **when** and **how** to call each one. Every tool validates its inputs and returns an error message instead of crashing, so the agent can explain the problem to the user.

In [3]:
# --- Tools the agent can call -------------------------------------------
# Each function has type hints and a docstring: the Gemini SDK reads them
# to tell the model what each tool does and which arguments it needs.

R_KPA_L = 8.314  # ideal gas constant in kPa·L/(mol·K)

# Factors to convert each unit to the base unit (Pa for pressure, kg for mass)
PRESSURE_TO_PA = {"pa": 1.0, "kpa": 1000.0, "bar": 1e5, "atm": 101325.0,
                  "psi": 6894.757, "mmhg": 133.322}
MASS_TO_KG = {"kg": 1.0, "g": 0.001, "lb": 0.45359237, "ton": 1000.0}
TEMP_UNITS = {"c", "f", "k"}


def _clean(unit: str) -> str:
    """Normalize a unit string: 'kPa' -> 'kpa', '°C' -> 'c'."""
    return unit.strip().lower().replace("°", "").replace(" ", "")


def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
    """Converts a value between units of temperature, pressure or mass.

    Supported units:
      Temperature: C, F, K
      Pressure: Pa, kPa, bar, atm, psi, mmHg
      Mass: kg, g, lb, ton

    Args:
      value: The number to convert.
      from_unit: The original unit, e.g. "atm".
      to_unit: The target unit, e.g. "kPa".
    """
    print(f"🔧 Tool called: convert_units({value}, '{from_unit}' -> '{to_unit}')")
    f, t = _clean(from_unit), _clean(to_unit)

    if f in TEMP_UNITS and t in TEMP_UNITS:
        kelvin = {"c": value + 273.15, "f": (value - 32) * 5 / 9 + 273.15, "k": value}[f]
        if kelvin < 0:
            return {"error": f"{value} {from_unit} is below absolute zero, which is physically impossible."}
        result = {"c": kelvin - 273.15, "f": (kelvin - 273.15) * 9 / 5 + 32, "k": kelvin}[t]
    elif f in PRESSURE_TO_PA and t in PRESSURE_TO_PA:
        result = value * PRESSURE_TO_PA[f] / PRESSURE_TO_PA[t]
    elif f in MASS_TO_KG and t in MASS_TO_KG:
        result = value * MASS_TO_KG[f] / MASS_TO_KG[t]
    else:
        return {"error": f"Cannot convert '{from_unit}' to '{to_unit}'. "
                         "Units must be supported and of the same type (temperature, pressure or mass)."}

    return {"value": round(result, 4), "unit": to_unit}


def ideal_gas_law(solve_for: str, pressure_kpa: float, volume_l: float,
                  moles: float, temperature_k: float) -> dict:
    """Solves the ideal gas law PV = nRT for one unknown variable.

    Pass 0 for the variable you want to solve for. Units MUST be
    kPa, liters, mol and Kelvin; convert other units first with convert_units.

    Args:
      solve_for: The unknown variable: "P", "V", "n" or "T".
      pressure_kpa: Pressure in kPa (0 if unknown).
      volume_l: Volume in liters (0 if unknown).
      moles: Amount of gas in mol (0 if unknown).
      temperature_k: Temperature in Kelvin (0 if unknown).
    """
    print(f"🔧 Tool called: ideal_gas_law(solve_for='{solve_for}', P={pressure_kpa}, "
          f"V={volume_l}, n={moles}, T={temperature_k})")
    known = {"P": pressure_kpa, "V": volume_l, "n": moles, "T": temperature_k}
    target = solve_for.strip().upper() if solve_for.strip().lower() != "n" else "n"

    if target not in known:
        return {"error": "solve_for must be one of: P, V, n, T."}
    for name, val in known.items():
        if name != target and val <= 0:
            return {"error": f"{name} must be a positive number (got {val})."}

    P, V, n, T = known["P"], known["V"], known["n"], known["T"]
    # Only compute the requested variable (computing all four would divide by the unknown 0)
    if target == "P":
        value, unit = n * R_KPA_L * T / V, "kPa"
    elif target == "V":
        value, unit = n * R_KPA_L * T / P, "L"
    elif target == "n":
        value, unit = P * V / (R_KPA_L * T), "mol"
    else:
        value, unit = P * V / (n * R_KPA_L), "K"
    return {"variable": target, "value": round(value, 4), "unit": unit}


def mixer_mass_balance(flows_kg_h: list[float], mass_fractions: list[float]) -> dict:
    """Steady-state mass balance for a mixer that combines several streams.

    Calculates the outlet flow and the outlet mass fraction of a component A.

    Args:
      flows_kg_h: Mass flow of each inlet stream in kg/h, e.g. [100, 50].
      mass_fractions: Mass fraction of component A in each inlet stream
        (between 0 and 1), in the same order, e.g. [0.2, 0.6].
    """
    print(f"🔧 Tool called: mixer_mass_balance(flows={flows_kg_h}, fractions={mass_fractions})")
    if len(flows_kg_h) != len(mass_fractions):
        return {"error": "Each stream needs exactly one flow and one mass fraction."}
    if any(f < 0 for f in flows_kg_h):
        return {"error": "Flows cannot be negative."}
    if any(not 0 <= x <= 1 for x in mass_fractions):
        return {"error": "Mass fractions must be between 0 and 1 (e.g. 40% = 0.4)."}

    total = sum(flows_kg_h)
    if total == 0:
        return {"error": "Total flow is zero, so there is nothing to mix."}
    mass_a = sum(f * x for f, x in zip(flows_kg_h, mass_fractions))
    return {"outlet_flow_kg_h": round(total, 4),
            "component_A_kg_h": round(mass_a, 4),
            "outlet_mass_fraction_A": round(mass_a / total, 4)}


### Test the tools without AI

Before connecting the model, check that each function gives correct results. (1 mol of gas at 0 °C and 1 atm should occupy ~22.4 L.)

In [4]:
print(convert_units(1, "atm", "kPa"))
print(convert_units(25, "C", "K"))
print(ideal_gas_law("V", pressure_kpa=101.325, volume_l=0, moles=1, temperature_k=273.15))
print(mixer_mass_balance([100, 50], [0.2, 0.6]))

# Edge cases: these should return clear errors, not crash
print(convert_units(-500, "C", "K"))
print(mixer_mass_balance([100], [40]))

🔧 Tool called: convert_units(1, 'atm' -> 'kPa')
{'value': 101.325, 'unit': 'kPa'}
🔧 Tool called: convert_units(25, 'C' -> 'K')
{'value': 298.15, 'unit': 'K'}
🔧 Tool called: ideal_gas_law(solve_for='V', P=101.325, V=0, n=1, T=273.15)
{'variable': 'V', 'value': 22.4127, 'unit': 'L'}
🔧 Tool called: mixer_mass_balance(flows=[100, 50], fractions=[0.2, 0.6])
{'outlet_flow_kg_h': 150, 'component_A_kg_h': 50.0, 'outlet_mass_fraction_A': 0.3333}
🔧 Tool called: convert_units(-500, 'C' -> 'K')
{'error': '-500 C is below absolute zero, which is physically impossible.'}
🔧 Tool called: mixer_mass_balance(flows=[100], fractions=[40])
{'error': 'Mass fractions must be between 0 and 1 (e.g. 40% = 0.4).'}


## 3. The agent

The *system instruction* defines the agent's behavior. Passing the Python functions in `tools` enables **automatic function calling**: the SDK runs the tool the model chooses and sends the result back to the model.

In [5]:
SYSTEM_PROMPT = """You are ChemE Assistant, a helpful chemical engineering tutor.

Rules:
- NEVER calculate numbers in your head. Always use the tools for any calculation.
- If units are not the ones a tool requires, call convert_units first.
- If the question is missing data, ask the user for it instead of guessing.
- If a tool returns an error, explain it simply and say how to fix the input.
- If the question is outside your tools, say what you can and cannot do.
- Answer in the same language as the user, step by step, and show units."""

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    tools=[convert_units, ideal_gas_law, mixer_mass_balance],
    temperature=0.2,  # low temperature = more consistent answers
)

chat = client.chats.create(model=MODEL, config=config)

import time
from IPython.display import display, Markdown

def ask(question: str, retries: int = 3):
    """Send a question to the agent and show its answer with formatted math.

    The free tier allows only a few requests per minute, so on a 429 error
    it waits and retries automatically instead of failing.
    """
    print(f"👤 {question}\n")
    for attempt in range(retries + 1):
        try:
            response = chat.send_message(question)
            print("\n🤖 Answer:")
            display(Markdown(response.text))  # renders headings, bold and LaTeX formulas
            print("-" * 70)
            return
        except Exception as e:
            if "429" in str(e) and attempt < retries:
                print(f"⏳ Free-tier limit reached. Waiting 60 s and retrying ({attempt + 1}/{retries})...")
                time.sleep(60)
            else:
                print(f"⚠️ Error: {str(e)[:300]}")
                return

## 4. Demo

Watch the `🔧 Tool called` lines: they show which tool the agent chose and with which arguments. Some questions need **more than one tool** (convert units first, then solve).

> The free tier allows only a few requests per minute, and each question uses 2–3 of them. If the limit is reached, `ask()` waits and retries on its own.

In [6]:
ask("¿Cuántos litros ocupan 2 moles de nitrógeno a 25 °C y 1.5 atm?")

👤 ¿Cuántos litros ocupan 2 moles de nitrógeno a 25 °C y 1.5 atm?

🔧 Tool called: convert_units(25, 'C' -> 'K')
🔧 Tool called: convert_units(1.5, 'atm' -> 'kPa')
🔧 Tool called: ideal_gas_law(solve_for='V', P=151.9875, V=0, n=2, T=298.15)

🤖 Answer:


Para calcular el volumen que ocupan 2 moles de nitrógeno a 25 °C y 1.5 atm, seguimos los siguientes pasos:

### Paso 1: Convertir las unidades a las requeridas ($K$ y $kPa$)
* **Temperatura:** 
  $$25 \text{ °C} = 298.15 \text{ K}$$
* **Presión:** 
  $$1.5 \text{ atm} = 151.9875 \text{ kPa}$$

### Paso 2: Aplicar la ley de los gases ideales ($P \cdot V = n \cdot R \cdot T$)
Sustituyendo los valores:
* $n = 2 \text{ mol}$
* $T = 298.15 \text{ K}$
* $P = 151.9875 \text{ kPa}$

Obtenemos un volumen de:
$$\mathbf{V \approx 32.62 \text{ litros}}$$

----------------------------------------------------------------------


In [7]:
ask("Mezclo 200 kg/h de una solución con 30% de etanol y 100 kg/h con 70% de etanol. ¿Cuál es la fracción de etanol a la salida?")

👤 Mezclo 200 kg/h de una solución con 30% de etanol y 100 kg/h con 70% de etanol. ¿Cuál es la fracción de etanol a la salida?

🔧 Tool called: mixer_mass_balance(flows=[200, 100], fractions=[0.3, 0.7])

🤖 Answer:


Para determinar la fracción mássica de etanol a la salida del mezclador, realizamos un balance de materia en estado estacionario:

### 1. Datos de las corrientes de entrada
* **Corriente 1:** $F_1 = 200 \text{ kg/h}$, con fracción mássica $x_1 = 0.30$ (30%)
* **Corriente 2:** $F_2 = 100 \text{ kg/h}$, con fracción mássica $x_2 = 0.70$ (70%)

### 2. Balance de masa total
$$F_{\text{salida}} = F_1 + F_2 = 200 \text{ kg/h} + 100 \text{ kg/h} = 300 \text{ kg/h}$$

### 3. Balance de masa para el etanol
$$\dot{m}_{\text{etanol}} = (200 \text{ kg/h} \times 0.30) + (100 \text{ kg/h} \times 0.70) = 60 \text{ kg/h} + 70 \text{ kg/h} = 130 \text{ kg/h}$$

### 4. Fracción mássica de etanol a la salida
$$x_{\text{etanol, salida}} = \frac{130 \text{ kg/h}}{300 \text{ kg/h}} \approx \mathbf{0.4333} \quad (\text{o } 43.33\%)$$

La fracción de etanol a la salida es **0.4333** (43.33% en masa).

----------------------------------------------------------------------


In [8]:
ask("Convert 350 °F to Celsius and 30 psi to bar.")

👤 Convert 350 °F to Celsius and 30 psi to bar.

⏳ Free-tier limit reached. Waiting 60 s and retrying (1/3)...
🔧 Tool called: convert_units(350, 'F' -> 'C')
🔧 Tool called: convert_units(30, 'psi' -> 'bar')

🤖 Answer:


Here are the conversions using standard unit conversion formulas:

### 1. Temperature Conversion (350 °F to °C)
* **Value:** $350 \text{ °F}$
* **Result:** **$176.67 \text{ °C}$**

### 2. Pressure Conversion (30 psi to bar)
* **Value:** $30 \text{ psi}$
* **Result:** **$2.0684 \text{ bar}$**

----------------------------------------------------------------------


## 5. Test, break and improve

Questions designed to make the agent fail: missing data, impossible values and requests outside its tools. The goal is to check that it asks for data or explains the error instead of inventing an answer.

In [ ]:
ask("¿Cuál es la presión de un gas a -300 °C?")  # below absolute zero

In [10]:
ask("¿Qué volumen ocupa un gas a 300 K?")  # missing data: pressure and moles

👤 ¿Qué volumen ocupa un gas a 300 K?


🤖 Answer:


Para calcular el volumen que ocupa el gas, falta información necesaria. Según la ley de los gases ideales ($P \cdot V = n \cdot R \cdot T$), necesito que me proporciones los siguientes datos:

1. **La cantidad de gas** (número de moles $n$ o la masa y el tipo de gas).
2. **La presión** a la que se encuentra el gas (por ejemplo, en atm, kPa, bar, etc.).

Por favor, proporciona estos valores para poder realizar el cálculo.

----------------------------------------------------------------------


In [11]:
ask("Calcula el diámetro de una tubería para 5 m/s.")  # no tool for this

👤 Calcula el diámetro de una tubería para 5 m/s.


🤖 Answer:


No dispongo de herramientas específicas para realizar cálculos de mecánica de fluidos o dimensionamiento de tuberías (mis herramientas disponibles están limitadas a conversiones de unidades, la ley de los gases ideales y balances de materia en mezcladores). 

Además, para calcular el diámetro de una tubería ($D$) a partir de la velocidad del fluido ($v = 5 \text{ m/s}$), se requiere conocer el **caudal volumétrico** ($Q$, por ejemplo en $\text{m}^3/\text{s}$ o $\text{L/min}$) o bien el caudal másico y la densidad del fluido, utilizando la relación:

$$Q = A \cdot v = \frac{\pi D^2}{4} \cdot v \implies D = \sqrt{\frac{4Q}{\pi v}}$$

Si necesitas ayuda con conversiones de unidades, cálculos de la ley de los gases ideales o balances de materia en un mezclador, con gusto puedo ayudarte.

----------------------------------------------------------------------


## 6. Try it yourself

Run this cell and chat with the agent. Type `salir` to stop.

In [ ]:
while True:
    q = input("Tu pregunta (o 'salir'): ")
    if q.strip().lower() in {"salir", "exit", "quit"}:
        break
    ask(q)

Tu pregunta (o 'salir'): ¿A que equivale 1 atm en mmHg?
👤 ¿A que equivale 1 atm en mmHg?

🔧 Tool called: convert_units(1, 'atm' -> 'mmHg')

🤖 Answer:


La conversión de $1 \text{ atm}$ a milímetros de mercurio ($\text{mmHg}$) da como resultado:

$$1 \text{ atm} = \mathbf{760.0021 \text{ mmHg}} \quad (\text{comúnmente redondeado a } 760 \text{ mmHg})$$

----------------------------------------------------------------------
Tu pregunta (o 'salir'): ¿Que es lo que necesita un balance de masa general?
👤 ¿Que es lo que necesita un balance de masa general?

⏳ Free-tier limit reached. Waiting 60 s and retrying (1/3)...

🤖 Answer:


Un **balance de materia general** es la aplicación del principio de conservación de la masa a un sistema definido. Para plantearlo y resolverlo correctamente, se requieren los siguientes elementos y componentes:

---

### 1. La Ecuación General de Balance
$$\text{Entrada} - \text{Salida} + \text{Generación} - \text{Consumo} = \text{Acumulación}$$

* **Entrada:** Masa que ingresa a través de los límites del sistema.
* **Salida:** Masa que sale a través de los límites del sistema.
* **Generación:** Masa producida dentro del sistema por reacción química.
* **Consumo:** Masa consumida dentro del sistema por reacción química.
* **Acumulación:** Cambio de masa dentro del sistema respecto al tiempo.

---

### 2. Información y Pasos Necesarios para Resolverlo

1. **Definición de los Límites del Sistema:**
   Determinar la unidad de proceso o conjunto de unidades sobre el cual se realiza el balance (volumen de control).

2. **Diagrama de Flujo (PFD):**
   Dibujar y etiquetar todas las corrientes de entrada y salida con sus variables conocidas y desconocidas.

3. **Base de Cálculo:**
   Elegir una cantidad o flujo de referencia (por ejemplo: $100 \text{ kg/h}$ o $1 \text{ kmol}$) si no se proporcionan valores absolutos.

4. **Variables de Corriente:**
   * **Flujos:** Másicos ($\dot{m}$) o molares ($\dot{n}$).
   * **Composiciones:** Fracciones másicas ($x_i$) o molares ($y_i$) de cada componente.

5. **Definición del Tipo de Proceso:**
   * **Estado Estacionario (*Steady State*):** Las propiedades no cambian con el tiempo ($\text{Acumulación} = 0$).
   * **Proceso sin Reacción Química:** No hay creación ni destrucción de especies ($\text{Generación} = 0, \text{Consumo} = 0$).

6. **Análisis de Grados de Libertad:**
   Verificar que el número de ecuaciones independientes (balances por componentes + relaciones adicionales) sea igual al número de variables desconocidas.

----------------------------------------------------------------------
Tu pregunta (o 'salir'): ¿Que dia es hoy?
👤 ¿Que dia es hoy?


🤖 Answer:


No tengo acceso a información en tiempo real ni a un reloj del sistema, por lo que no puedo decirte la fecha o el día de hoy. 

Mis herramientas están enfocadas en ingeniería química para realizar:
* Conversiones de unidades (temperatura, presión, masa).
* Cálculos de la ley de los gases ideales ($P \cdot V = n \cdot R \cdot T$).
* Balances de masa en mezcladores.

Si necesitas ayuda con alguno de estos temas, ¡con gusto te ayudo!

----------------------------------------------------------------------
Tu pregunta (o 'salir'): Hablame sobre procesos de separacion
👤 Hablame sobre procesos de separacion

⏳ Free-tier limit reached. Waiting 60 s and retrying (1/3)...
⏳ Free-tier limit reached. Waiting 60 s and retrying (2/3)...
⏳ Free-tier limit reached. Waiting 60 s and retrying (3/3)...


## 7. What I learned

Things broke while building this, and fixing them taught me the most:

- **Bug in my own tool.** The first version of `ideal_gas_law` computed P, V, n and T at once, so it divided by the unknown variable (0) and crashed. I changed it to compute only the requested variable and added input validation.
- **Model retired (404).** The model I first hard-coded was no longer available for new users, so every request failed. I made the notebook list the models available for the API key and pick a working one automatically.
- **Rate limits (429).** Running all cells at once exceeded the free-tier quota, because each question uses 2–3 API calls. I added automatic waiting and retries.
- **Unreadable output.** Answers came back with raw LaTeX and Markdown symbols. I rendered them as Markdown so formulas display correctly.
- **Tool descriptions are prompts.** The model decides which tool to use only from the function names, type hints and docstrings, so writing them clearly matters as much as the code.

**Next steps:** more tools (heat exchanger energy balance, Antoine equation for vapor pressure) and a simple web interface.